# Adult Census Income — Feature Engineering

This notebook applies **feature engineering techniques only** (no model training) on the [Adult Census Income](https://archive.ics.uci.edu/ml/datasets/adult) dataset.

**Dataset path:** `c:\Users\user\OneDrive\Desktop\Dataset\adult.csv`

**Target variable:** `income` (`<=50K` or `>50K`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    StandardScaler, MinMaxScaler, RobustScaler,
    PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

DATA_PATH = r'c:\Users\user\OneDrive\Desktop\Dataset\adult.csv'
OUTPUT_DIR = 'processed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

---
## Step 1: Load & Explore Data

Load the CSV, inspect shape, dtypes, missing values (`?`), and target distribution.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDtypes:\n{df.dtypes}')
print(f'\nMissing values (?):\n{(df == "?").sum()}')
print(f'\nTarget distribution:\n{df["income"].value_counts()}')
df.head()

In [ ]:
df.describe(include='all').T

---
## Step 2: Handle Missing Values

Replace `?` with `NaN`, then impute:
- **Categorical columns** → mode (most frequent value)
- **Numerical columns** → median

In [ ]:
df = df.replace('?', np.nan)

cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=np.number).columns.tolist()

cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer = SimpleImputer(strategy='median')

df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
df[num_cols] = num_imputer.fit_transform(df[num_cols])

print('Missing values after imputation:', df.isnull().sum().sum())
df.to_csv(f'{OUTPUT_DIR}/step2_missing_handled.csv', index=False)
df.head()

---
## Step 3: Drop Irrelevant Features

Remove columns that do not help prediction:
- **`fnlwgt`** — sampling weight from census, not a predictive feature
- **`education`** — redundant with `education.num` (ordinal numeric version)

In [ ]:
df = df.drop(columns=['fnlwgt', 'education'])
print(f'Shape after dropping: {df.shape}')
print(f'Remaining columns: {list(df.columns)}')
df.to_csv(f'{OUTPUT_DIR}/step3_dropped_features.csv', index=False)

---
## Step 4: Domain Feature Creation

Create new features from domain knowledge:
- **`net_capital`** = capital.gain − capital.loss
- **`has_capital_gain`** = 1 if capital.gain > 0, else 0
- **`has_capital_loss`** = 1 if capital.loss > 0, else 0
- **`work_intensity`** = hours.per.week / 40 (full-time ratio)
- **`is_full_time`** = 1 if hours.per.week ≥ 35

In [ ]:
df['net_capital'] = df['capital.gain'] - df['capital.loss']
df['has_capital_gain'] = (df['capital.gain'] > 0).astype(int)
df['has_capital_loss'] = (df['capital.loss'] > 0).astype(int)
df['work_intensity'] = df['hours.per.week'] / 40
df['is_full_time'] = (df['hours.per.week'] >= 35).astype(int)

print('New features created:')
print(df[['net_capital', 'has_capital_gain', 'has_capital_loss', 'work_intensity', 'is_full_time']].describe())
df.to_csv(f'{OUTPUT_DIR}/step4_domain_features.csv', index=False)

---
## Step 5: Binning / Discretization

Convert continuous variables into categorical bins:
- **`age_group`** — Young / Middle / Senior / Elderly
- **`hours_group`** — Part-time / Full-time / Overtime

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 40, 55, 100],
    labels=['Young', 'Middle', 'Senior', 'Elderly']
)

df['hours_group'] = pd.cut(
    df['hours.per.week'],
    bins=[0, 20, 40, 60, 100],
    labels=['Part-time', 'Full-time', 'Overtime', 'Heavy-overtime']
)

print('Age group distribution:')
print(df['age_group'].value_counts())
print('\nHours group distribution:')
print(df['hours_group'].value_counts())
df.to_csv(f'{OUTPUT_DIR}/step5_binned_features.csv', index=False)

---
## Step 6: Log Transformation

Apply `log1p` (log(1+x)) to highly skewed numeric features to reduce skewness:
- `capital.gain`, `capital.loss`, `net_capital`

In [ ]:
skewed_cols = ['capital.gain', 'capital.loss', 'net_capital']

for col in skewed_cols:
    df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['capital.gain'].hist(bins=50, ax=axes[0])
axes[0].set_title('capital.gain (original)')
df['capital.gain_log'].hist(bins=50, ax=axes[1])
axes[1].set_title('capital.gain_log (transformed)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/step6_log_transform.png', dpi=100)
plt.show()

df.to_csv(f'{OUTPUT_DIR}/step6_log_transformed.csv', index=False)

---
## Step 7: Outlier Treatment (IQR Capping)

Cap extreme values using the Interquartile Range (IQR) method on numeric columns.

In [ ]:
def cap_outliers_iqr(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return series.clip(lower=lower, upper=upper)

outlier_cols = ['age', 'capital.gain', 'capital.loss', 'hours.per.week', 'education.num']

for col in outlier_cols:
    before_max = df[col].max()
    df[f'{col}_capped'] = cap_outliers_iqr(df[col])
    print(f'{col}: max {before_max:.0f} -> {df[f"{col}_capped"].max():.0f}')

df.to_csv(f'{OUTPUT_DIR}/step7_outlier_capped.csv', index=False)

---
## Step 8: Label Encoding

Convert binary/low-cardinality categorical columns to integers:
- `sex` → Male=1, Female=0
- `income` (target) → >50K=1, <=50K=0

In [ ]:
le = LabelEncoder()

df['sex_encoded'] = le.fit_transform(df['sex'])
df['income_encoded'] = le.fit_transform(df['income'])

print('Sex encoding:', dict(zip(le.classes_, le.transform(le.classes_))))
print('Income encoding:', dict(zip(le.classes_, le.transform(le.classes_))))
print(df[['sex', 'sex_encoded', 'income', 'income_encoded']].head())
df.to_csv(f'{OUTPUT_DIR}/step8_label_encoded.csv', index=False)

---
## Step 9: One-Hot Encoding

Convert nominal categorical variables into binary dummy columns:
- `workclass`, `marital.status`, `occupation`, `relationship`, `race`, `age_group`, `hours_group`

In [ ]:
onehot_cols = ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'age_group', 'hours_group']

df_onehot = pd.get_dummies(df, columns=onehot_cols, prefix=onehot_cols, drop_first=True)

print(f'Shape before one-hot: {df.shape}')
print(f'Shape after one-hot:  {df_onehot.shape}')
print(f'\nNew columns added: {df_onehot.shape[1] - df.shape[1]}')
df_onehot.to_csv(f'{OUTPUT_DIR}/step9_onehot_encoded.csv', index=False)
df_onehot.head()

---
## Step 10: Ordinal Encoding

`education.num` is already ordinal (1=lowest … 16=highest). We also encode `native.country` by frequency rank.

In [ ]:
country_freq = df['native.country'].value_counts(normalize=True)
df['country_freq_encoded'] = df['native.country'].map(country_freq)

country_rank = {country: rank for rank, country in enumerate(country_freq.index, 1)}
df['country_rank_encoded'] = df['native.country'].map(country_rank)

print('Top 5 countries by frequency:')
print(country_freq.head())
print(f'\neducation.num range: {df["education.num"].min()} - {df["education.num"].max()}')
df.to_csv(f'{OUTPUT_DIR}/step10_ordinal_encoded.csv', index=False)

---
## Step 11: Frequency Encoding

Replace high-cardinality categories with their occurrence frequency in the dataset (useful for `native.country` with 40+ unique values).

In [ ]:
for col in ['native.country', 'occupation', 'workclass']:
    freq_map = df[col].value_counts(normalize=True).to_dict()
    df[f'{col}_freq'] = df[col].map(freq_map)

print('Frequency-encoded columns:')
print(df[['native.country', 'native.country_freq', 'occupation_freq', 'workclass_freq']].head())
df.to_csv(f'{OUTPUT_DIR}/step11_frequency_encoded.csv', index=False)

---
## Step 12: Feature Scaling

Scale numeric features so they have comparable ranges. Compare three scalers:
- **StandardScaler** — zero mean, unit variance
- **MinMaxScaler** — scale to [0, 1]
- **RobustScaler** — uses median & IQR (robust to outliers)

In [ ]:
scale_cols = ['age', 'education.num', 'capital.gain', 'capital.loss',
              'hours.per.week', 'net_capital', 'work_intensity']

X = df[scale_cols].copy()
y = df['income_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

scaled_results = {}
for name, scaler in scalers.items():
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    scaled_results[name] = pd.DataFrame(X_train_scaled, columns=scale_cols)
    print(f'\n{name} — train set stats:')
    print(scaled_results[name].describe().round(2))

X_train_std = pd.DataFrame(scalers['StandardScaler'].fit_transform(X_train), columns=scale_cols)
X_test_std = pd.DataFrame(scalers['StandardScaler'].transform(X_test), columns=scale_cols)
X_train_std.to_csv(f'{OUTPUT_DIR}/step12_train_scaled.csv', index=False)
X_test_std.to_csv(f'{OUTPUT_DIR}/step12_test_scaled.csv', index=False)

---
## Step 13: Polynomial Features

Generate interaction terms and polynomial features (degree=2) for selected numeric columns.

In [ ]:
poly_cols = ['age', 'education.num', 'hours.per.week']

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df[poly_cols])
poly_feature_names = poly.get_feature_names_out(poly_cols)

df_poly = pd.DataFrame(X_poly, columns=poly_feature_names)
print(f'Original features: {len(poly_cols)}')
print(f'Polynomial features: {df_poly.shape[1]}')
print(f'\nFeature names: {list(poly_feature_names)}')
df_poly.head()

---
## Step 14: Feature Selection

Select the top-K most informative features using:
- **Chi-squared (χ²)** test on non-negative features
- **Mutual Information** for numeric features vs. target

In [ ]:
feature_cols = ['age', 'education.num', 'capital.gain', 'capital.loss',
                'hours.per.week', 'net_capital', 'work_intensity',
                'has_capital_gain', 'has_capital_loss', 'is_full_time',
                'sex_encoded', 'country_freq_encoded']

X_sel = df[feature_cols].copy()
X_sel = X_sel - X_sel.min()  # chi2 requires non-negative values
y_sel = df['income_encoded']

mi_scores = mutual_info_classif(X_sel, y_sel, random_state=42)
mi_df = pd.DataFrame({'feature': feature_cols, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)

print('Mutual Information scores:')
print(mi_df.to_string(index=False))

selector = SelectKBest(score_func=chi2, k=8)
selector.fit(X_sel, y_sel)
chi2_scores = pd.DataFrame({
    'feature': feature_cols,
    'chi2_score': selector.scores_
}).sort_values('chi2_score', ascending=False)

print('\nChi-squared scores:')
print(chi2_scores.to_string(index=False))

selected = chi2_scores.head(8)['feature'].tolist()
print(f'\nTop 8 selected features: {selected}')

---
## Step 15: Correlation Analysis

Visualize correlations between numeric features and the target to detect multicollinearity.

In [ ]:
corr_cols = feature_cols + ['income_encoded']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/step15_correlation_matrix.png', dpi=100)
plt.show()

print('Correlation with income:')
print(corr_matrix['income_encoded'].sort_values(ascending=False).to_string())

---
## Step 16: Final Feature-Engineered Dataset

Combine the best transformations into a final modeling-ready dataset.

In [ ]:
final_features = [
    'age', 'education.num', 'capital.gain_log', 'capital.loss_log',
    'hours.per.week', 'net_capital_log', 'work_intensity', 'is_full_time',
    'has_capital_gain', 'has_capital_loss', 'sex_encoded',
    'country_freq_encoded', 'income_encoded'
]

df_final = df[final_features].copy()
df_final.to_csv(f'{OUTPUT_DIR}/final_feature_engineered.csv', index=False)

print(f'Final dataset shape: {df_final.shape}')
print(f'\nFeatures: {list(df_final.columns)}')
print(f'\nDescriptive statistics:')
df_final.describe().round(2)

---
## Summary

| Step | Technique | Output File |
|------|-----------|-------------|
| 1 | Load & Explore | — |
| 2 | Missing Value Handling | `step2_missing_handled.csv` |
| 3 | Drop Irrelevant Features | `step3_dropped_features.csv` |
| 4 | Domain Feature Creation | `step4_domain_features.csv` |
| 5 | Binning / Discretization | `step5_binned_features.csv` |
| 6 | Log Transformation | `step6_log_transformed.csv` |
| 7 | Outlier Treatment (IQR) | `step7_outlier_capped.csv` |
| 8 | Label Encoding | `step8_label_encoded.csv` |
| 9 | One-Hot Encoding | `step9_onehot_encoded.csv` |
| 10 | Ordinal Encoding | `step10_ordinal_encoded.csv` |
| 11 | Frequency Encoding | `step11_frequency_encoded.csv` |
| 12 | Feature Scaling | `step12_train/test_scaled.csv` |
| 13 | Polynomial Features | (in-memory) |
| 14 | Feature Selection | (scores printed) |
| 15 | Correlation Analysis | `step15_correlation_matrix.png` |
| 16 | Final Dataset | `final_feature_engineered.csv` |